# ComplaintIQ: Qwen3-4B QLoRA training on Windows

Runs the same scripts as the `run_*.bat` files, step by step, with live output.
Open this with `start_notebook.bat` (it uses the kit's `.venv`). Everything is offline.

**Order for the day:** 1 environment check → 2 dry run → 3 speed test → 4 train → 5 evaluate → 6 export GGUF → 7 copy back.

> Long training is safer from `run_train.bat` in a normal window (a browser tab closing will not stop it there).
> If the kernel dies mid-training, just run step 4 again: it resumes from the last checkpoint.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path
from IPython.display import Markdown, display

KIT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
RUN = "qwen3-4b-main"            # results folder name for the main run
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false",
                  BITSANDBYTES_NOWELCOME="1", PYTHONUTF8="1", PYTHONUNBUFFERED="1")

def run(script, *args):
    """Run a kit script in a child process (Windows-safe) and stream its output."""
    cmd = [sys.executable, "-u", str(KIT / "src" / script), *map(str, args)]
    print(">", " ".join(cmd), flush=True)
    proc = subprocess.Popen(cmd, cwd=KIT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        print(line, end="", flush=True)
    if proc.wait() != 0:
        raise RuntimeError(f"{script} failed with exit code {proc.returncode}")

print("Kit:", KIT, "| Python:", sys.executable)

## 1. Environment check
CUDA, bf16, a real bitsandbytes 4-bit forward/backward, model/data files, llama.cpp tools. Must end with `READY`.

In [ ]:
run("env_check.py")

## 2. Dry run (~5-10 min)
Loads Qwen3-4B in 4-bit, 5 training steps on 64 rows, saves and reloads the adapter, evaluates 32 rows. Must end with `DRY RUN PASSED`.

If bitsandbytes fails here and you cannot fix it within ~30 minutes, switch to the WSL2 fallback in the README.

In [ ]:
run("train_qwen3.py", "--mode", "dryrun")

## 3. Speed test (~200 steps)
Measures seconds/step and projects the full-epoch time. Use the recommendation to choose step 4's settings.

In [ ]:
run("train_qwen3.py", "--mode", "speedtest")
speed = json.loads((KIT / "results" / "speedtest-Qwen3-4B-Instruct-2507" / "speedtest.json").read_text())
{k: speed[k] for k in ["seconds_per_step", "tokens_per_second", "peak_vram_gb", "projected_hours_per_epoch", "steps_that_fit_in_6h", "steps_that_fit_in_7h", "recommendation"]}

## 4. Train
Default: 1 epoch with a 7-hour safety budget (stops cleanly and saves). Checkpoints every 250 steps, validation loss + accuracy every 500 steps (`train_log.jsonl`). Re-running resumes automatically.

If the speed test said a full epoch does not fit, set `EXTRA = ["--max-steps", "<steps_that_fit_in_7h>"]`.

In [ ]:
EXTRA = []   # e.g. ["--max-steps", "3500"] or ["--time-budget-hours", "6"]
run("train_qwen3.py", "--mode", "train", "--run-name", RUN, "--time-budget-hours", "7", *EXTRA)

In [ ]:
# Progress at any time (also works while training runs from run_train.bat)
log = KIT / "results" / RUN / "train_log.jsonl"
events = [json.loads(l) for l in log.read_text(encoding="utf-8").splitlines()] if log.exists() else []
for e in [e for e in events if e["event"] in ("val_accuracy", "time_budget_reached", "done")][-10:]:
    print(e)
losses = [(e["step"], round(e["loss"], 4)) for e in events if e["event"] == "log" and "loss" in e]
print("last losses:", losses[-10:])

## 5. Evaluate
Training already evaluates the final adapter on the test and check sets. The cell below shows the Qwen3-4B results (`eval\\results.md`, also saved as `results.json`). Set `ADAPTER` to a checkpoint to evaluate that instead.

In [ ]:
ADAPTER = KIT / "results" / RUN / "final_adapter"     # or .../checkpoints/checkpoint-NNNN
if ADAPTER.name != "final_adapter" or not (KIT / "results" / RUN / "eval" / "results.md").exists():
    run("evaluate_qwen3.py", "--adapter", ADAPTER)
results_dir = KIT / "results" / RUN / ("eval" if ADAPTER.name == "final_adapter" else f"eval_{ADAPTER.name}/eval")
display(Markdown((results_dir / "results.md").read_text(encoding="utf-8")))

## 6. Export GGUF for the app (~20-30 min)
Merges the adapter into the bf16 base (CPU, ~10 GB RAM), converts with llama.cpp, quantizes to Q4_K_M (~2.5 GB), then runs 300 test rows through `llama-server.exe` with the app's exact request to confirm the GGUF behaves like the adapter.

In [ ]:
run("export_gguf.py", "--adapter", KIT / "results" / RUN / "final_adapter", "--verify-rows", "300")

## 7. Copy back
Close this notebook, then run in a terminal from the kit folder:

```
copy_back.bat E:\complaintiq_results
```

It copies `results\` (adapters, GGUF, evaluations, logs, newest checkpoint) and skips bulky intermediates. See README section 7.